# C1 · Perfect separation: when the logistic becomes very sure and wrong

**Type**: Break the model · **Level**: ●●○ independent · **Time**: about 40 minutes · **core path ★**

## Objective

Build a case in which the classes are **perfectly separable on the training set** (few examples, many features), observe that without regularization the weights explode and the probabilities
become extreme, and show that the **AUC does not notice** while the log-loss does. Then find `C` with cross-validation.

## Dataset

`generate()` (given): 60 training examples and 5000 test ones with 60 features of which only 5 informative (`make_classification`, with 5% of the labels swapped).
With more features than examples there is always a hyperplane that separates the training set, even if the features say almost nothing.

## Educational purpose

See live three things that often happen in real data: (1) with `p > n` (or small classes) the training set is separable and the unregularized logistic **has no
minimum**; (2) its "confidence" is an artifact: predictions at 99% and wrong; (3) the ranking metrics (AUC) are blind to all this.

## Task

1. For `C` in `Cs = np.logspace(-4, 6, 11)` train `StandardScaler` + `LogisticRegression(C=C, max_iter=20000)` on the training set and build `table` (index `C`, columns `train_acc`, `l1_norm` (sum of the absolute values of the coefficients), `train_logloss`, `test_logloss`, `test_auc`, `confident_share` (fraction of test predictions with `p < 0.01` or `p > 0.99`), `confident_errors` (of these, how many are **wrong**, using 0.5 as threshold).
2. Save `best_test_C` (the `C` with the minimum test log-loss: the absolute best, which you usually don't know).
3. Choose `C` **without** looking at the test set: `GridSearchCV` with `scoring="neg_log_loss"` and `StratifiedKFold(5, shuffle=True, random_state=0)` on the training set, on the same pipeline and on the `Cs`. Save `cv_C` (float) and check that the corresponding test log-loss is much better than the one without regularization.
4. Save `auc_range = table['test_auc'].max() - table['test_auc'].min()` and `logloss_range` (the same for the test log-loss).

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


def generate():
    X, y = make_classification(n_samples=5060, n_features=60, n_informative=5, n_redundant=0, class_sep=1.0, flip_y=0.05, random_state=1)
    return X[:60], y[:60], X[60:], y[60:]


X_tr, y_tr, X_te, y_te = generate()
Cs = np.logspace(-4, 6, 11)

## Your work

In [ ]:
rows = []
for c in Cs:
    ...
table = pd.DataFrame(rows, index=Cs)
table.round(3)

In [ ]:
best_test_C = None
cv_C = None
auc_range = None
logloss_range = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='For every `c`: `m = make_pipeline(StandardScaler(), LogisticRegression(C=c, max_iter=20000)).fit(X_tr, y_tr)`; `p = np.clip(m.predict_proba(X_te)[:, 1], 1e-15, 1 - 1e-15)`; `confident = (p < 0.01) | (p > 0.99)`; `errors = (p > 0.5) != y_te`. `cv_C = float(GridSearchCV(m, {"logisticregression__C": Cs}, ...).fit(X_tr, y_tr).best_params_["logisticregression__C"])`.'):
    check(list(table.columns) == ["train_acc", "l1_norm", "train_logloss", "test_logloss", "test_auc", "confident_share", "confident_errors"], "table with the seven columns", "check names and order")
    check(table.loc[Cs[-1], "train_acc"] == 1.0 and table.loc[Cs[0], "train_acc"] < 0.7, "without regularization the training set is perfectly separated", "train_acc with C = 1e6 must be 1")
    check(table["l1_norm"].is_monotonic_increasing, "the norm of the weights always grows with C", "more C, less penalty, more weights")
    check(table.loc[Cs[-1], "train_logloss"] < 1e-3 and table.loc[Cs[-1], "test_logloss"] > 5, "training: log-loss almost 0; test: over 5", "train_logloss and test_logloss at the row C = 1e6")
    check(table.loc[Cs[-1], "confident_errors"] > 1000, "over 1000 examples out of 5000 are wrong with probability below 1% or above 99%", "count (p < 0.01 or p > 0.99) and (predicted class != y)")
    check(best_test_C == table["test_logloss"].idxmin() and best_test_C <= 0.1, "the best C on the test set is small (strong regularization)", "idxmin of the test log-loss")
    check(isinstance(cv_C, float) and cv_C <= 0.1 and table.loc[cv_C, "test_logloss"] < 0.75, "the C chosen in cross-validation clearly beats the unpenalized logistic on the test set", "GridSearchCV with scoring='neg_log_loss'")
    check(auc_range < 0.06 and logloss_range > 6, "the test AUC varies little (less than 0.06), the log-loss by more than 6", "compare the two ranges")
done()

---
The commented solution is in `solutions/C_break_the_model/C1_perfect_separation_and_extreme_probabilities.ipynb`. Open it after you pass the check, or after you have really tried.